# 02: Data Collection

Fetch mental health articles from PubMed. MIMIC-IV requires credentialed access, so we use PubMed as primary source.

In [2]:
import os
import sys

# 1. Setup paths and mount Google Drive if in Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Define and change to project root
    PROJECT_ROOT = '/content/drive/MyDrive/mental_health_rag_kg'
    os.chdir(PROJECT_ROOT)
    sys.path.append(PROJECT_ROOT)
    print(f"Running in Colab. Project root set to: {os.getcwd()}")
else:
    # Local environment
    # If running from notebooks/, go up one level to root
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    
    PROJECT_ROOT = os.getcwd()
    sys.path.append(PROJECT_ROOT)
    print(f"Running locally. Project root set to: {os.getcwd()}")

from rag.pubmed_fetcher import PubMedFetcher
from dotenv import load_dotenv
load_dotenv()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab. Project root set to: /content/drive/MyDrive/mental_health_rag_kg


True

In [3]:
# Initialize fetcher
fetcher = PubMedFetcher(
    api_key=os.getenv('PUBMED_API_KEY'),
    email='rsriad00@gmail.com'  # REQUIRED by NCBI
)

In [4]:
# Define mental health search queries
queries = [
    'mental health disorders symptoms treatment',
    'depression anxiety diagnosis management',
    'PTSD OCD schizophrenia clinical guidelines',
    'bipolar disorder mood swings therapy',
    'insomnia sleep disorders cognitive behavioral therapy'
]

all_files = []
for q in queries:
    fpath = fetcher.fetch_and_save(
        query=q,
        output_dir='data/raw/pubmed',
        max_results=20
    )
    if fpath:
        all_files.append(fpath)
    print('---')

Searching PubMed for: mental health disorders symptoms treatment
Found 20 articles.
Fetched 20 abstracts with text.
---
Searching PubMed for: depression anxiety diagnosis management
Found 20 articles.
Fetched 20 abstracts with text.
---
Searching PubMed for: PTSD OCD schizophrenia clinical guidelines
Found 3 articles.
Fetched 3 abstracts with text.
---
Searching PubMed for: bipolar disorder mood swings therapy
Found 20 articles.
Fetched 19 abstracts with text.
---
Searching PubMed for: insomnia sleep disorders cognitive behavioral therapy
Found 20 articles.
Fetched 20 abstracts with text.
---


In [5]:
# Combine all PubMed files into one
import json
from pathlib import Path

combined = []
for fpath in all_files:
    with open(fpath, 'r') as f:
        for line in f:
            combined.append(json.loads(line))

out_path = 'data/raw/pubmed/all_articles.jsonl'
Path(out_path).parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    for item in combined:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"File saved to: {Path(out_path).resolve()}")
print(f'Total articles collected: {len(combined)}')


# out_path = 'data/raw/pubmed/all_articles.jsonl'
# Path(out_path).parent.mkdir(parents=True, exist_ok=True)
# total_articles = 0

# with open(out_path, 'w', encoding='utf-8') as f_out:
#     for fpath in all_files:
#         with open(fpath, 'r', encoding='utf-8') as f_in:
#             for line in f_in:
#                 # Optional: validate JSON before writing
#                 item = json.loads(line)
#                 f_out.write(json.dumps(item, ensure_ascii=False) + '\n')
#                 total_articles += 1

# print(f'Total articles collected: {total_articles}')

File saved to: /content/drive/MyDrive/mental_health_rag_kg/data/raw/pubmed/all_articles.jsonl
Total articles collected: 100
